In [30]:
import numpy as np
import cv2
import glob
import imutils


In [31]:
image_paths = glob.glob('./images/*.jpeg')
images = []

for index, image in enumerate(image_paths):
    img = cv2.imread(image)
    images.append(img)
    # cv2.imshow(f"Image {index+1}", img)
    # cv2.waitKey(0)


# PostProcessing (not working yet):

In [45]:
def find_crop_bounds(thresh_img, edge_check_pct=0.15):
    """
    edge_check_pct: % da altura a verificar nas bordas das colunas (topo e base)
    """
    h, w = thresh_img.shape

    # ── 1. TOPO: primeira linha sem nenhum pixel preto ────────────────────────
    top_y = 0
    for row in range(h):
        if np.all(thresh_img[row, :] > 0):
            top_y = row
            break

    # ── 2. BASE: última linha sem nenhum pixel preto ──────────────────────────
    bottom_y = h
    for row in range(h - 1, -1, -1):
        if np.all(thresh_img[row, :] > 0):
            bottom_y = row + 1
            break

    thresh_cropped = thresh_img[top_y:bottom_y, :]
    ch = bottom_y - top_y

    # Quantas linhas verificar nas bordas das colunas
    edge_px = max(1, int(ch * edge_check_pct))

    def col_has_black_on_edges(col_data):
        """Retorna True se a coluna tem preto no topo OU na base"""
        top_edge = col_data[:edge_px]
        bot_edge = col_data[-edge_px:]
        return np.any(top_edge == 0) or np.any(bot_edge == 0)

    # ── 3. LATERAL ESQUERDA ───────────────────────────────────────────────────
    left_x = 0
    for col in range(w):
        if not col_has_black_on_edges(thresh_cropped[:, col]):
            left_x = col
            break

    # ── 4. LATERAL DIREITA ────────────────────────────────────────────────────
    right_x = w
    for col in range(w - 1, -1, -1):
        if not col_has_black_on_edges(thresh_cropped[:, col]):
            right_x = col + 1
            break

    return left_x, top_y, right_x - left_x, bottom_y - top_y


In [46]:
imageStitcher = cv2.Stitcher_create()

error, stitched_image = imageStitcher.stitch(images)

if not error:
    cv2.imwrite("stitchedImage.png", stitched_image)

    stitched_img = cv2.copyMakeBorder(stitched_image, 10, 10, 10, 10, cv2.BORDER_CONSTANT, (0,0,0))

    gray = cv2.cvtColor(stitched_image, cv2.COLOR_BGR2GRAY)
    thresh_img = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY)[1]

    contours = cv2.findContours(thresh_img.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = imutils.grab_contours(contours)

    if len(contours) == 0:
        print("Nenhum contorno encontrado no thresh_img")
        exit()

    areaOI = max(contours, key=cv2.contourArea)

    mask = np.zeros(thresh_img.shape, dtype="uint8")
    x, y, w, h = find_crop_bounds(thresh_img)
    cv2.rectangle(mask, (x, y), (x + w, y + h), 255, -1)

    minRectangle = mask.copy()
    sub = mask.copy()

    MAX_ITERATIONS = 65
    iterations = 0

    while cv2.countNonZero(sub) > 0 and iterations < MAX_ITERATIONS:
        eroded = cv2.erode(minRectangle, None)
        
        if cv2.countNonZero(eroded) == 0:
            print("Erosão zeraria a imagem — interrompendo")
            break
        
        minRectangle = eroded
        sub = cv2.subtract(minRectangle, thresh_img)
        iterations += 1

    contours = cv2.findContours(minRectangle.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = imutils.grab_contours(contours)

    if len(contours) == 0:
        print("Fallback: usando contorno do thresh_img diretamente")
        contours = cv2.findContours(thresh_img.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contours = imutils.grab_contours(contours)

    if len(contours) == 0:
        print("Nenhum contorno encontrado — abortando")
        exit()

    areaOI = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(areaOI)

    stitched_img = stitched_img[y:y + h, x:x + w]
    cv2.imwrite("stitchedOutputProcessed_MyAlgo.png", stitched_img)

else:
    print(f"Error: {error}")
    